# Migration: Target ODI SQL to Databricks Spark SQL
**Source File:** target_file.sql
**Conversion Date:** 2023-11-01
**Description:** Conversion of ODI session tasks for HR Jobs data loading.

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "")

### ETL Parameters initialization (SCEN_TASK_NO {1})

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_params AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'job_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS etl_last_run_dt;

### Database Link Cleanup (SCEN_TASK_NO {30}, {60})
Note: Oracle Database Links are not applicable in Databricks. External sources are accessed via Mount Points, Unity Catalog Volumes, or JDBC connectors.

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Database link removal - Not applicable in Spark SQL environment;
-- SCEN_TASK_NO {60}: Database link creation - Not applicable in Spark SQL environment;

### Insert into Target (SCEN_TASK_NO {80})

In [ ]:
%sql
INSERT INTO workspace.odi_trg.trg_jobs
(
    job_id,
    job_title,
    min_salary,
    max_salary,
    last_upd_dt
)
SELECT
    src_jobs.job_id,
    src_jobs.job_title,
    src_jobs.min_salary,
    src_jobs.max_salary,
    src_jobs.last_upd_dt
FROM workspace.odi_src.src_jobs AS src_jobs
WHERE
    src_jobs.last_upd_dt >= to_date('${v_lst_dt}', 'dd-MM-yy');

### Final Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.odi_trg.trg_jobs;